# Final Project: Reproducing Multi-Output SCA Neural Network Results

## Overview
This notebook reproduces the results from Hoang et al. (2023): "Efficient Nonprofiled Side-Channel Attack Using Multi-Output Classification Neural Network"

The paper introduces:
- **MLPMO**: Multi-Output MLP for masking and noise-generation countermeasures
- **CNNMO**: Multi-Output CNN for de-synchronization countermeasures

Both models can predict all 256 key hypotheses in a single training process, achieving 9-30x speedup over DDLA.

In [ ]:
# Import required libraries
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
from pathlib import Path

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import our custom modules
import sys
sys.path.append('.')

from labeling import create_multi_output_labels_batch
from data_loader import (
    load_ascad_dataset, add_gaussian_noise, apply_desynchronization,
    create_dataset_variants, create_noisy_datasets, create_desync_datasets
)
from models import MLPMO, CNNMO
from training import train_model, evaluate_accuracy, identify_correct_key
from evaluation import (
    plot_accuracy_curves, plot_all_key_accuracies, plot_attack_time_comparison,
    plot_success_rate_comparison, calculate_success_rate, create_comparison_table
)

## Dataset Preparation

**Note**: This section requires the ASCAD dataset. If you don't have it yet, you can:
1. Download from: https://github.com/ANSSI-FR/ASCAD
2. Place the HDF5 file in `labs/Final_project/datasets/`

For now, we'll create a placeholder structure that can be used once the dataset is available.

In [ ]:
# Dataset configuration
DATASET_DIR = Path('datasets')
ASCAD_PATH = DATASET_DIR / 'ASCAD.h5'  # Update this path to your ASCAD dataset location

# Check if dataset exists
if ASCAD_PATH.exists():
    print(f"Found ASCAD dataset at {ASCAD_PATH}")
    # Load dataset
    # traces, plaintexts, keys, metadata = load_ascad_dataset(str(ASCAD_PATH), group='Profiling_traces')
    # print(f"Loaded {metadata['num_traces']} traces with length {metadata['trace_length']}")
else:
    print(f"ASCAD dataset not found at {ASCAD_PATH}")
    print("Please download the ASCAD dataset and update ASCAD_PATH")
    print("For now, we'll create a synthetic dataset structure for demonstration")
    
    # Create synthetic data structure for testing (will be replaced with real data)
    SYNTHETIC_MODE = True

## Create PyTorch Dataset Class

In [ ]:
class SCADataset(Dataset):
    """
    PyTorch Dataset for side-channel attack data.
    """
    def __init__(self, traces, plaintexts, keys, correct_key_byte_idx=2):
        """
        Args:
            traces: Power traces (N, trace_length)
            plaintexts: Plaintext bytes (N,)
            keys: Key bytes (N,)
            correct_key_byte_idx: Index of the key byte to attack (default: 2 for third Sbox)
        """
        self.traces = torch.FloatTensor(traces)
        self.plaintexts = plaintexts.astype(np.uint8)
        self.keys = keys.astype(np.uint8)
        self.correct_key_byte_idx = correct_key_byte_idx
        
        # Get correct key byte value
        self.correct_key = int(self.keys[0, correct_key_byte_idx])
        
        # Create multi-output labels
        # Extract the relevant plaintext byte (assuming same index as key byte)
        plaintext_bytes = self.plaintexts[:, correct_key_byte_idx]
        self.labels = create_multi_output_labels_batch(plaintext_bytes, num_key_guesses=256)
        self.labels = torch.LongTensor(self.labels)
    
    def __len__(self):
        return len(self.traces)
    
    def __getitem__(self, idx):
        return self.traces[idx], self.labels[idx]

## Model Implementation

### MLPMO (Multi-Output MLP)

**Decision**: MLP chosen for masking/noise countermeasures because:
- MLP architecture is effective for fixed-alignment power traces
- Temporal relationships are less critical for these countermeasures
- Simpler architecture allows faster training

In [ ]:
# Test MLPMO model creation
# Note: This will be used once we have real data
# For now, we'll define the trace length based on typical ASCAD values

TRACE_LENGTH = 700  # Typical ASCAD trace length (will be updated from actual data)

# Create MLPMO variants
mlpmo_non_sosl = MLPMO(input_size=TRACE_LENGTH, shared_layer_size=0, num_key_guesses=256)
mlpmo_sosl_200 = MLPMO(input_size=TRACE_LENGTH, shared_layer_size=200, num_key_guesses=256)

print("MLPMO Non-SoSL model created")
print(f"  Total parameters: {sum(p.numel() for p in mlpmo_non_sosl.parameters()):,}")
print("\nMLPMO SoSL-200 model created")
print(f"  Total parameters: {sum(p.numel() for p in mlpmo_sosl_200.parameters()):,}")

# Move to device
mlpmo_non_sosl = mlpmo_non_sosl.to(device)
mlpmo_sosl_200 = mlpmo_sosl_200.to(device)

### CNNMO (Multi-Output CNN)

**Decision**: CNN chosen for de-synchronization countermeasures because:
- CNN's translation-invariance property makes it robust to random timing shifts
- Convolutional layers can learn patterns regardless of their position in the trace
- This is essential for handling de-synchronized power traces

In [ ]:
# Create CNNMO model
cnnmo = CNNMO(input_size=TRACE_LENGTH, num_key_guesses=256, num_filters=64, kernel_size=3)

print("CNNMO model created")
print(f"  Total parameters: {sum(p.numel() for p in cnnmo.parameters()):,}")

# Move to device
cnnmo = cnnmo.to(device)

## Experiment 1: Masking Countermeasure (MLPMO)

This experiment reproduces the masking countermeasure results using ASCAD Dataset2.
We compare Non-SoSL, SoSL-200, and (optionally) MLPDDLA baseline.

In [ ]:
# Placeholder for masking experiment
# This will be implemented once the dataset is loaded

print("Masking countermeasure experiment")
print("=" * 50)
print("This experiment requires:")
print("1. ASCAD dataset loaded")
print("2. Dataset2 created (20,000 traces)")
print("3. Multi-output labels generated")
print("4. Training of Non-SoSL and SoSL-200 models")
print("5. Comparison of attack times and accuracy gaps")
print()
print("Once dataset is available, uncomment and run the following:")
print()
print("# Load and prepare Dataset2")
print("# traces, plaintexts, keys, metadata = load_ascad_dataset(str(ASCAD_PATH))")
print("# datasets = create_dataset_variants(traces, plaintexts, keys)")
print("# dataset2 = datasets['Dataset2']")
print()
print("# Create dataset and dataloader")
print("# sca_dataset = SCADataset(dataset2['traces'], dataset2['plaintexts'], dataset2['keys'])")
print("# train_loader = DataLoader(sca_dataset, batch_size=128, shuffle=True)")
print()
print("# Train Non-SoSL model")
print("# history_non_sosl = train_model(")
print("#     mlpmo_non_sosl, train_loader, train_loader, num_epochs=30, device=device,")
print("#     correct_key=sca_dataset.correct_key, verbose=True")
print("# )")

## Experiment 2: Noise Generation Countermeasure

This experiment tests the models' robustness to additive Gaussian noise with varying σ levels.

In [ ]:
# Placeholder for noise generation experiment
print("Noise generation countermeasure experiment")
print("=" * 50)
print("This experiment requires:")
print("1. ASCAD dataset loaded")
print("2. Noisy datasets created (DatasetX-N1, N2, N3 with σ = 0.5, 1.0, 1.5)")
print("3. Multiple attack runs (50 runs per configuration)")
print("4. Success rate calculation")
print()
print("Expected results:")
print("- All models achieve 100% success rate at σ = 0.5")
print("- SoSL-200 should outperform MLPDDLA by 20%+ at σ = 1.0 and 1.5")

## Experiment 3: De-Synchronization Countermeasure (CNNMO)

This experiment tests CNNMO's ability to handle de-synchronized traces.

In [ ]:
# Placeholder for de-synchronization experiment
print("De-synchronization countermeasure experiment")
print("=" * 50)
print("This experiment requires:")
print("1. ASCAD dataset loaded")
print("2. De-synchronized datasets created (random shifts up to 20 samples)")
print("3. CNNMO model training")
print("4. Comparison with CNNDDLA baseline (if implemented)")
print()
print("Expected results:")
print("- CNNMO should achieve ~30x speedup over CNNDDLA")
print("- Clear key discrimination from early epochs")

## Results and Evaluation

Once experiments are complete, this section will contain:
- Accuracy plots showing correct key discrimination
- Attack time comparison tables
- Success rate results for noisy data
- Training logs and metrics

In [ ]:
# Placeholder for results visualization
print("Results visualization will be generated here once experiments are complete")
print()
print("Planned visualizations:")
print("1. Accuracy curves (correct key vs incorrect keys)")
print("2. Attack time comparison bar charts")
print("3. Success rate vs noise level plots")
print("4. Accuracy per key hypothesis plots")